### Load and call data again

In [1]:
import pandas as pd

file_path = "../data/raw/Akwaaba_Insurance_Dataset.xlsx"

# Load all sheets into a dictionary of DataFrames
sheets = pd.read_excel(file_path, sheet_name=None)
sheets.keys()
for name in sheets.keys():
    print(name)




Corporate Clients
Covered Employees
Claims
Policy Table
Premium Payments
Branch Table
Claim Workflow
Beneficiaries
Risk Scores


In [2]:
### Loading all tables of the dataset

corporate_clients = sheets["Corporate Clients"]
covered_employees = sheets["Covered Employees"]
claims = sheets["Claims"]
policy_table = sheets["Policy Table"]
premium_payments = sheets["Premium Payments"]
branch_table = sheets["Branch Table"]
claim_workflow = sheets["Claim Workflow"]
beneficiaries = sheets["Beneficiaries"]
risk_scores = sheets["Risk Scores"]

### Data Cleaning 01 - Corporate Clients

In [3]:

## Checking the first few rows, the structure, duplicate records and missing values.

print(corporate_clients.head().to_string())

print(corporate_clients.info())

print(corporate_clients.duplicated().sum())

print(corporate_clients.isnull().sum())

  Corporate_ID Company_Name Branch_ID            Industry    Region  Number_of_Staff             Insurance_Product  Premium_Per_Staff_GHS Insurance_Purpose Application_Date
0      CORP001    MTN Ghana     BR001  Telecommunications     Accra             1800            Group Welfare Plan                    248       HR Benefits       2025-11-15
1      CORP002    MTN Ghana     BR002  Telecommunications    Kumasi              250            Group Welfare Plan                    254  Employee Welfare       2024-11-08
2      CORP003    MTN Ghana     BR003  Telecommunications  Takoradi              120  Group Funeral Insurance Plan                    219  Talent Retention       2024-05-24
3      CORP004    MTN Ghana     BR004  Telecommunications    Tamale               80            Group Welfare Plan                    232  Employee Welfare       2024-11-06
4      CORP005    MTN Ghana     BR005  Telecommunications        Ho               35     Group Life Insurance Plan                    2

In [4]:
# ============================================
# Split Corporate Clients into companies + corporate_policies table
# ============================================

#create new companies table with only Company Name and Industry and creating a new ID for each company
companies = corporate_clients[['Company_Name', 'Industry']].drop_duplicates().reset_index(drop=True)
companies['Company_ID'] = ['COMP' + str(i+1).zfill(3) for i in range(len(companies))]
companies = companies[['Company_ID', 'Company_Name', 'Industry']]

#Building new corporate_policies table from corporate_clients table and merging with companies table
corporate_policies = corporate_clients.merge(
    companies[['Company_ID', 'Company_Name']],
    on='Company_Name',
    how='left'
)

corporate_policies = corporate_policies[[
    'Corporate_ID', 'Company_ID', 'Branch_ID', 'Number_of_Staff',
    'Insurance_Product', 'Premium_Per_Staff_GHS', 'Insurance_Purpose', 'Application_Date'
]]

# Convert date column to proper datetime
corporate_policies['Application_Date'] = pd.to_datetime(corporate_policies['Application_Date'])

In [5]:
# rename "Number_of_Staff" to "Estimated_Headcount"
corporate_policies = corporate_policies.rename(columns={'Number_of_Staff': 'Estimated_Headcount'})

### Note on renaming Number_of_Staff → Estimated_Headcount

During cleaning, we tested whether `Number_of_Staff` matched the actual number of employees enrolled 
under each branch's plan (from `Covered Employees`). It did not — in many cases, by a large margin 
(e.g. one branch declared 1800 staff, but only 32 employees were actually enrolled).

This shows the column is not a real headcount of the branch — it's a number declared by the company 
at the time of application, likely used to estimate pricing and plan for future enrollment growth. 
The original name "Number_of_Staff" implied a verified fact about the branch, which was misleading.

We renamed it to `Estimated_Headcount` to better reflect what it actually represents: a declared estimate 
used for pricing, not a confirmed count of people covered.

In [6]:
print(corporate_policies.columns.tolist())

['Corporate_ID', 'Company_ID', 'Branch_ID', 'Estimated_Headcount', 'Insurance_Product', 'Premium_Per_Staff_GHS', 'Insurance_Purpose', 'Application_Date']


In [7]:
#Sanity check


print(corporate_policies.head())
print(corporate_policies.head().dtypes)

print('companies shape:', companies.shape)          # expect (7, 3)

print('corporate_policies shape:', corporate_policies.shape)  # expect (70, 9)

print('Any null Company_ID after merge:', corporate_policies['Company_ID'].isnull().sum())  # expect 0

print(companies.head())
print(companies.head().dtypes)


  Corporate_ID Company_ID Branch_ID  Estimated_Headcount  \
0      CORP001    COMP001     BR001                 1800   
1      CORP002    COMP001     BR002                  250   
2      CORP003    COMP001     BR003                  120   
3      CORP004    COMP001     BR004                   80   
4      CORP005    COMP001     BR005                   35   

              Insurance_Product  Premium_Per_Staff_GHS Insurance_Purpose  \
0            Group Welfare Plan                    248       HR Benefits   
1            Group Welfare Plan                    254  Employee Welfare   
2  Group Funeral Insurance Plan                    219  Talent Retention   
3            Group Welfare Plan                    232  Employee Welfare   
4     Group Life Insurance Plan                    226  Employee Welfare   

  Application_Date  
0       2025-11-15  
1       2024-11-08  
2       2024-05-24  
3       2024-11-06  
4       2024-03-06  
Corporate_ID                        str
Company_ID      

### Data Cleaning 02 - Covered Employees

In [8]:
covered_employees.head()

,Employee_ID,Corporate_ID,Employee_Name,Occupation,Grade_Level,Monthly_Salary_GHS,Age,Gender,Dependents,Smoker
0,EMP00001,CORP001,April Perez,Network Engineer,Mid,12000,43,Male,2,Yes
1,EMP00002,CORP001,Jonathan Morris,Software Engineer,Junior,8500,40,Female,4,No
2,EMP00003,CORP001,Kathryn Cook,Network Engineer,Junior,7500,40,Male,4,No
3,EMP00004,CORP001,Christopher Johnson,Software Engineer,Mid,13000,51,Female,0,Yes
4,EMP00005,CORP001,Kenneth Thompson,Software Engineer,Junior,8500,58,Female,0,No


In [9]:
## Get the unique titles and suffixes found in the Employee Name column under the covered_employee table



ce = sheets["Covered Employees"]
names = ce['Employee_Name']

titles_found = set()
suffixes_found = set()

for name in names:
    words = name.split()
    
    if len(words) == 2:
        continue  # normal "First Last" name, nothing to extract
    
    # A title sits BEFORE the real name (e.g. "Mr. John Smith" -> "Mr." is word[0])
    # We detect it by checking if the first word ends in a period, or is "Miss"
    if words[0].endswith('.') or words[0] == 'Miss':
        titles_found.add(words[0])
        words = words[1:]  # remove the title, what's left should be First Last (+maybe a suffix)
    
    # After removing a title (if any), anything beyond the first 2 words is a suffix
    if len(words) > 2:
        suffixes_found.update(words[2:])

print('Unique titles found:', sorted(titles_found))
print('Unique suffixes found:', sorted(suffixes_found))

Unique titles found: ['Dr.', 'Miss', 'Mr.', 'Mrs.', 'Ms.']
Unique suffixes found: ['DDS', 'DVM', 'Jr.', 'MD', 'PhD']


In [10]:
#Extract Title and suffix from covered_employee table and clean it, separating into new columns 

titles = ['Dr.', 'Mr.', 'Mrs.', 'Ms.', 'Miss']
suffixes = ['MD', 'DVM', 'DDS', 'PhD', 'Jr.']

def extract_title(name):
    first_word = name.split()[0]
    return first_word if first_word in titles else None

def extract_suffix(name):
    last_word = name.split()[-1]
    return last_word if last_word in suffixes else None

def clean_name(name):
    words = name.split()
    if words[0] in titles:
        words = words[1:]
    if words[-1] in suffixes:
        words = words[:-1]
    return ' '.join(words)

covered_employees['Employee_Title'] = covered_employees['Employee_Name'].apply(extract_title)
covered_employees['Employee_Suffix'] = covered_employees['Employee_Name'].apply(extract_suffix)
covered_employees['Employee_Name'] = covered_employees['Employee_Name'].apply(clean_name)

print(covered_employees[['Employee_Name', 'Employee_Title', 'Employee_Suffix']].head(15).to_string())
print()
print('Title counts:')
print(covered_employees['Employee_Title'].value_counts())
print()
print('Suffix counts:')
print(covered_employees['Employee_Suffix'].value_counts())

          Employee_Name Employee_Title Employee_Suffix
0           April Perez            NaN             NaN
1       Jonathan Morris            NaN             NaN
2          Kathryn Cook            NaN             NaN
3   Christopher Johnson            NaN             NaN
4      Kenneth Thompson            NaN             NaN
5            Wendy Kidd            NaN             NaN
6            John Smith            NaN             NaN
7      Timothy Callahan            NaN             NaN
8     Jessica Blanchard            NaN             NaN
9         Fernando Lara            NaN             NaN
10      Patricia Duncan            NaN             NaN
11       Melissa Cannon            NaN             NaN
12          Keith Allen            NaN             NaN
13       Brenda Griffin            NaN             NaN
14        Donna Johnson            NaN             NaN

Title counts:
Employee_Title
Mr.     14
Mrs.    11
Dr.      9
Miss     2
Ms.      1
Name: count, dtype: int64

Suffix c

In [11]:
import gender_guesser.detector as gd

d = gd.Detector()

def get_first_name(name):
    return name.split()[0]

covered_employees['First_Name_Temp'] = covered_employees['Employee_Name'].apply(get_first_name)

covered_employees['Gender_Guess'] = covered_employees['First_Name_Temp'].apply(
    lambda x: d.get_gender(x)
)

gender_map = {
    'male': 'Male',
    'mostly_male': 'Male',
    'female': 'Female',
    'mostly_female': 'Female',
    'andy': 'Unknown',
    'unknown': 'Unknown'
}
covered_employees['Gender_Guess'] = covered_employees['Gender_Guess'].map(gender_map)

covered_employees = covered_employees.drop(columns=['First_Name_Temp'])

print(covered_employees[['Employee_Name', 'Gender', 'Gender_Guess']].head(10))
print(covered_employees['Gender_Guess'].value_counts())

         Employee_Name  Gender Gender_Guess
0          April Perez    Male       Female
1      Jonathan Morris  Female         Male
2         Kathryn Cook    Male       Female
3  Christopher Johnson  Female         Male
4     Kenneth Thompson  Female         Male
5           Wendy Kidd  Female       Female
6           John Smith    Male         Male
7     Timothy Callahan  Female         Male
8    Jessica Blanchard    Male       Female
9        Fernando Lara  Female         Male
Gender_Guess
Male       997
Female     971
Unknown      6
Name: count, dtype: int64


### Note on Gender and Gender_Guess columns

During profiling, we tested whether the `Gender` column aligns with the first names in `Employee_Name`. 
Using a name-based gender lookup, only ~50.5% of confidently-gendered names matched the recorded `Gender` value — 
statistically equivalent to random chance (50/50). This indicates `Gender` was likely assigned independently 
of `Employee_Name` during data generation, not derived from it.

Since there is no reliable ground truth to "correct" this, the original `Gender` column is kept unchanged.

`Gender_Guess` is an additional, clearly separate column inferred purely from first names, added for 
exploratory/personalization purposes only. It is **not** a replacement or correction of `Gender`, 
and the two columns are expected to disagree roughly half the time.

In [12]:
covered_employees = covered_employees[[
  'Employee_ID', 'Corporate_ID', 'Employee_Name', 'Employee_Title', 'Employee_Suffix', 'Occupation', 
  'Grade_Level', 'Monthly_Salary_GHS', 'Age', 'Gender', 'Gender_Guess', 'Dependents', 'Smoker',
]]
print(covered_employees.columns.tolist())

['Employee_ID', 'Corporate_ID', 'Employee_Name', 'Employee_Title', 'Employee_Suffix', 'Occupation', 'Grade_Level', 'Monthly_Salary_GHS', 'Age', 'Gender', 'Gender_Guess', 'Dependents', 'Smoker']


In [13]:
covered_employees.head()

,Employee_ID,Corporate_ID,Employee_Name,Employee_Title,Employee_Suffix,Occupation,Grade_Level,Monthly_Salary_GHS,Age,Gender,Gender_Guess,Dependents,Smoker
0,EMP00001,CORP001,April Perez,NaN,NaN,Network Engineer,Mid,12000,43,Male,Female,2,Yes
1,EMP00002,CORP001,Jonathan Morris,NaN,NaN,Software Engineer,Junior,8500,40,Female,Male,4,No
2,EMP00003,CORP001,Kathryn Cook,NaN,NaN,Network Engineer,Junior,7500,40,Male,Female,4,No
3,EMP00004,CORP001,Christopher Johnson,NaN,NaN,Software Engineer,Mid,13000,51,Female,Male,0,Yes
4,EMP00005,CORP001,Kenneth Thompson,NaN,NaN,Software Engineer,Junior,8500,58,Female,Male,0,No


### Data Cleaning 03 - Claims Table

In [14]:
print(claims.columns.tolist())

['Claim_ID', 'Employee_ID', 'Corporate_ID', 'Claim_Type', 'Claim_Amount_GHS', 'Claim_Status', 'Claim_Date']


In [15]:
# Dropping Corporate ID from Claims Table because it is redundant, since Claims table is derivable via Employee ID -> Covered Employees -> Policy ID
claims = claims.drop(columns=['Corporate_ID'])

# Convert Claim_Date from text to proper datetime
claims['Claim_Date'] = pd.to_datetime(claims['Claim_Date'])
print(claims.dtypes)

Claim_ID                       str
Employee_ID                    str
Claim_Type                     str
Claim_Amount_GHS             int64
Claim_Status                   str
Claim_Date          datetime64[us]
dtype: object


In [16]:
claims.head()

,Claim_ID,Employee_ID,Claim_Type,Claim_Amount_GHS,Claim_Status,Claim_Date
0,CLM00001,EMP01950,Funeral Benefit,15040,Approved,2024-09-24
1,CLM00002,EMP00184,Dental,7155,Rejected,2025-12-08
2,CLM00003,EMP00706,Funeral Benefit,15572,Pending,2025-03-07
3,CLM00004,EMP01925,Hospitalization,44381,Approved,2024-05-12
4,CLM00005,EMP01390,Emergency Care,14988,Pending,2024-07-28


### Data Cleaning 04 - Policy Table

In [17]:
policy_table.head()

,Policy_ID,Corporate_ID,Insurance_Product,Policy_Start_Date,Policy_End_Date,Policy_Status
0,POL0001,CORP001,Group Welfare Plan,2025-11-15,2026-11-15,Renewed
1,POL0002,CORP002,Group Welfare Plan,2024-11-08,2025-11-08,Renewed
2,POL0003,CORP003,Group Funeral Insurance Plan,2024-05-24,2025-05-24,Pending Renewal
3,POL0004,CORP004,Group Welfare Plan,2024-11-06,2025-11-06,Renewed
4,POL0005,CORP005,Group Life Insurance Plan,2024-03-06,2025-03-06,Pending Renewal


In [18]:
## Dropping Insurance Product - 100% Duplicate of what's already in corporate_policies
policy_table = policy_table.drop(columns=['Insurance_Product'])

# Convert date columns from text to proper datetime
policy_table['Policy_Start_Date'] = pd.to_datetime(policy_table['Policy_Start_Date'])
policy_table['Policy_End_Date'] = pd.to_datetime(policy_table['Policy_End_Date'])

print(policy_table.dtypes)

Policy_ID                       str
Corporate_ID                    str
Policy_Start_Date    datetime64[us]
Policy_End_Date      datetime64[us]
Policy_Status                   str
dtype: object


In [19]:
policy_table.head()

,Policy_ID,Corporate_ID,Policy_Start_Date,Policy_End_Date,Policy_Status
0,POL0001,CORP001,2025-11-15,2026-11-15,Renewed
1,POL0002,CORP002,2024-11-08,2025-11-08,Renewed
2,POL0003,CORP003,2024-05-24,2025-05-24,Pending Renewal
3,POL0004,CORP004,2024-11-06,2025-11-06,Renewed
4,POL0005,CORP005,2024-03-06,2025-03-06,Pending Renewal


### Data Cleaning 05 - Premium Payments Table

In [20]:
premium_payments.head()

,Payment_ID,Corporate_ID,Payment_Date,Amount_GHS,Payment_Status,Payment_Method
0,PAY00011,CORP001,2025-12-15,37200,Pending,Payroll Deduction
1,PAY00012,CORP001,2026-01-15,37200,Paid,Bank Transfer
2,PAY00013,CORP001,2026-02-15,37200,Paid,Payroll Deduction
3,PAY00014,CORP001,2026-03-15,37200,Paid,Bank Transfer
4,PAY00015,CORP001,2026-04-15,37200,Paid,Bank Transfer


In [21]:
# Convert Payment_Date from text to proper datetime
premium_payments['Payment_Date'] = pd.to_datetime(premium_payments['Payment_Date'])

print(premium_payments.dtypes)

Payment_ID                   str
Corporate_ID                 str
Payment_Date      datetime64[us]
Amount_GHS                 int64
Payment_Status               str
Payment_Method               str
dtype: object


### Data Cleaning 06 - Branches Table

In [22]:
branch_table.head()

,Branch_ID,Company_Name,Region
0,BR001,MTN Ghana,Accra
1,BR002,MTN Ghana,Kumasi
2,BR003,MTN Ghana,Takoradi
3,BR004,MTN Ghana,Tamale
4,BR005,MTN Ghana,Ho


In [23]:
# Replace Company_Name with Company_ID, linking to the companies table
branches = branch_table.merge(companies[['Company_ID', 'Company_Name']], on='Company_Name', how='left')

# Keep only Branch_ID, Company_ID, Region
branches = branches[['Branch_ID', 'Company_ID', 'Region']]


In [24]:
print(branches.head(10))
print(branches.dtypes)
print('Shape:', branches.shape)
print('Nulls:', branches['Company_ID'].isnull().sum())

  Branch_ID Company_ID    Region
0     BR001    COMP001     Accra
1     BR002    COMP001    Kumasi
2     BR003    COMP001  Takoradi
3     BR004    COMP001    Tamale
4     BR005    COMP001        Ho
5     BR006    COMP002     Accra
6     BR007    COMP002    Kumasi
7     BR008    COMP002  Takoradi
8     BR009    COMP002    Tamale
9     BR010    COMP002        Ho
Branch_ID     str
Company_ID    str
Region        str
dtype: object
Shape: (70, 3)
Nulls: 0


### Data Cleaning 07 - Claim Workflow Table

In [25]:
# Convert date columns to proper datetime
# Payout_Date keeps its nulls -- they represent "no payout recorded", not a data error

claim_workflow['Submitted_Date'] = pd.to_datetime(claim_workflow['Submitted_Date'])
claim_workflow['Reviewed_Date'] = pd.to_datetime(claim_workflow['Reviewed_Date'])
claim_workflow['Payout_Date'] = pd.to_datetime(claim_workflow['Payout_Date'])

print(claim_workflow.dtypes)
print(claim_workflow.isnull().sum())

Workflow_ID                  str
Claim_ID                     str
Submitted_Date    datetime64[us]
Reviewed_Date     datetime64[us]
Approved_By                  str
Payout_Date       datetime64[us]
dtype: object
Workflow_ID        0
Claim_ID           0
Submitted_Date     0
Reviewed_Date      0
Approved_By        0
Payout_Date       84
dtype: int64


### Note on Claim Workflow coverage and missing Payout_Date

`Claim Workflow` only contains 165 records, while `Claims` has 2,500 — meaning roughly 93% of claims 
have no workflow record at all. We checked whether this gap follows a pattern (e.g. only certain 
claim statuses get tracked), but all three statuses (Approved, Pending, Rejected) appear both with 
and without a workflow record in similar proportions. This appears to be a genuine data gap, not a 
designed rule, and cannot be filled in or inferred.

Similarly, 84 of the 165 workflow records have a missing `Payout_Date`. We checked whether this 
correlates with `Claim_Status` (e.g. assuming Approved claims should always have a payout date), 
but the data doesn't support that — even Approved claims are missing Payout_Date about as often as 
they have one. So missing `Payout_Date` values are kept as real nulls (meaning "no payout recorded"), 
rather than filled in with assumed or calculated dates.

Implication for analysis: any workflow-based metrics (e.g. average review time, average payout delay) 
should be understood as describing only this small 165-claim sample, not the full Claims dataset.

### Data Cleaning 07 - Beneficiaries Table

In [26]:
beneficiaries.head()

,Beneficiary_ID,Employee_ID,Beneficiary_Name,Relationship,Allocation_Percentage
0,BEN00001,EMP00001,April Perez Family,Spouse,100
1,BEN00002,EMP00002,Jonathan Morris Family,Sibling,100
2,BEN00003,EMP00003,Kathryn Cook Family,Parent,100
3,BEN00004,EMP00004,Christopher Johnson Family,Sibling,100
4,BEN00005,EMP00005,Kenneth Thompson Family,Sibling,100


### Note on Beneficiaries data limitations

Two structural issues were identified in this table, both of which appear to be limitations in how 
the dataset was generated, rather than something that can be cleaned or corrected:

**1. Beneficiary_Name is not a genuine distinct individual.**
Every `Beneficiary_Name` is the employee's own name with the word "Family" appended (e.g. "April Perez 
Family"), rather than the name of an actual separate person (spouse, parent, child, etc). This appears 
to be placeholder/templated data, not real beneficiary records. We cannot reconstruct genuine beneficiary 
names from this, so the column is retained as-is and flagged here as unreliable for any analysis that 
depends on knowing who the beneficiary actually is.

**2. Allocation_Percentage carries no analytical signal, and the data structure assumes exactly one 
beneficiary per employee.**
`Allocation_Percentage` is 100 for all 1,974 records, with zero variation. Investigation confirmed 
this is because the dataset enforces a strict 1-to-1 relationship: every employee has exactly one 
beneficiary, no more, no fewer. Real-world policies typically allow:
- Multiple beneficiaries with split allocations (e.g. 60/40 between a spouse and a child)
- Zero beneficiaries, if an employee has not designated one

Neither scenario is represented anywhere in this dataset. This is a meaningful business risk, not 
just a data quality footnote: if real-world data behaved this way, it could indicate the system has 
no way to handle employees without a named beneficiary, or to support legitimate multi-beneficiary 
payout splits, at claim/payout time.

**Recommendation:** this finding should be surfaced beyond the cleaning notebook — e.g. in a project 
findings/risks summary — since it reflects a potential gap in how the business captures beneficiary 
designations, not merely a formatting issue.

### Data Cleaning 09 - Risk Score Table

In [27]:
risk_scores.head()

,Risk_ID,Employee_ID
0,RSK00001,EMP00001
1,RSK00002,EMP00002
2,RSK00003,EMP00003
3,RSK00004,EMP00004
4,RSK00005,EMP00005


### Note on Risk Scores table — currently a placeholder

This table currently contains only `Risk_ID` and `Employee_ID` — no actual risk score, risk level, or 
risk category exists yet. There is nothing to clean here (no nulls, duplicates, or formatting issues), 
because there is no real data yet to have those problems.

This is expected, not an error: this table is the placeholder that the upcoming Feature Engineering 
phase (Notebook 3) is meant to fill in. A real risk score will need to be calculated using signals 
from other tables already cleaned — e.g. Age, Smoker status, Monthly_Salary_GHS, Claim_Amount_GHS, 
Claim frequency, and similar factors.

No transformation is applied to this table in Notebook 2. It is carried forward as-is, to be populated 
during feature engineering.

In [28]:
cleaned_tables = {
    'companies': companies,
    'corporate_policies': corporate_policies,
    'covered_employees': covered_employees,
    'claims': claims,
    'policy_table': policy_table,
    'premium_payments': premium_payments,
    'branches': branches,
    'claim_workflow': claim_workflow,
    'beneficiaries': sheets["Beneficiaries"],   # unchanged, no cleaning applied
    'risk_scores': sheets["Risk Scores"]         # unchanged, placeholder table
}

for name, df in cleaned_tables.items():
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print()

--- companies ---
Shape: (7, 3)
Columns: ['Company_ID', 'Company_Name', 'Industry']

--- corporate_policies ---
Shape: (70, 8)
Columns: ['Corporate_ID', 'Company_ID', 'Branch_ID', 'Estimated_Headcount', 'Insurance_Product', 'Premium_Per_Staff_GHS', 'Insurance_Purpose', 'Application_Date']

--- covered_employees ---
Shape: (1974, 13)
Columns: ['Employee_ID', 'Corporate_ID', 'Employee_Name', 'Employee_Title', 'Employee_Suffix', 'Occupation', 'Grade_Level', 'Monthly_Salary_GHS', 'Age', 'Gender', 'Gender_Guess', 'Dependents', 'Smoker']

--- claims ---
Shape: (2500, 6)
Columns: ['Claim_ID', 'Employee_ID', 'Claim_Type', 'Claim_Amount_GHS', 'Claim_Status', 'Claim_Date']

--- policy_table ---
Shape: (70, 5)
Columns: ['Policy_ID', 'Corporate_ID', 'Policy_Start_Date', 'Policy_End_Date', 'Policy_Status']

--- premium_payments ---
Shape: (420, 6)
Columns: ['Payment_ID', 'Corporate_ID', 'Payment_Date', 'Amount_GHS', 'Payment_Status', 'Payment_Method']

--- branches ---
Shape: (70, 3)
Columns: ['Bra

In [29]:
NewTable = {
    'companies': companies,
    'corporate_policies': corporate_policies,
    'covered_employees': covered_employees,
    'claims': claims,
    'policy_table': policy_table,
    'premium_payments': premium_payments,
    'branches': branches,
    'claim_workflow': claim_workflow,
    'beneficiaries': sheets["Beneficiaries"]  
}
for key, value in cleaned_tables.items():
    print(key)
    print(value.columns)

companies
Index(['Company_ID', 'Company_Name', 'Industry'], dtype='str')
corporate_policies
Index(['Corporate_ID', 'Company_ID', 'Branch_ID', 'Estimated_Headcount',
       'Insurance_Product', 'Premium_Per_Staff_GHS', 'Insurance_Purpose',
       'Application_Date'],
      dtype='str')
covered_employees
Index(['Employee_ID', 'Corporate_ID', 'Employee_Name', 'Employee_Title',
       'Employee_Suffix', 'Occupation', 'Grade_Level', 'Monthly_Salary_GHS',
       'Age', 'Gender', 'Gender_Guess', 'Dependents', 'Smoker'],
      dtype='str')
claims
Index(['Claim_ID', 'Employee_ID', 'Claim_Type', 'Claim_Amount_GHS',
       'Claim_Status', 'Claim_Date'],
      dtype='str')
policy_table
Index(['Policy_ID', 'Corporate_ID', 'Policy_Start_Date', 'Policy_End_Date',
       'Policy_Status'],
      dtype='str')
premium_payments
Index(['Payment_ID', 'Corporate_ID', 'Payment_Date', 'Amount_GHS',
       'Payment_Status', 'Payment_Method'],
      dtype='str')
branches
Index(['Branch_ID', 'Company_ID', 'Region

### Exporting cleaned data to CSV from Notebook 2 and store in processed data folder in AHICP folder → load CSVs in Notebook 03 

In [30]:
import os

# Make sure the processed folder exists
os.makedirs("../data/processed", exist_ok=True)

# Export each cleaned table to CSV
for name, df in cleaned_tables.items():
    if name == "risk_scores":
        continue  # not loading this yet, still a placeholder
    df.to_csv(f"../data/processed/{name}.csv", index=False)
    print(f"Saved {name}.csv -- shape {df.shape}")

Saved companies.csv -- shape (7, 3)
Saved corporate_policies.csv -- shape (70, 8)
Saved covered_employees.csv -- shape (1974, 13)
Saved claims.csv -- shape (2500, 6)
Saved policy_table.csv -- shape (70, 5)
Saved premium_payments.csv -- shape (420, 6)
Saved branches.csv -- shape (70, 3)
Saved claim_workflow.csv -- shape (165, 6)
Saved beneficiaries.csv -- shape (1974, 5)
